In [15]:
import torch
import torchvision
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from sklearn.metrics import f1_score as sklearn_f1_score
from sklearn.metrics import precision_score, recall_score, accuracy_score

import torch.nn as nn
import torch.optim as optim
import copy

import tqdm

import sys, os, random
sys.path.append(os.path.join(os.getcwd(), 'CONFOLD')) #add CONFOLD to path

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, KFold, cross_val_score, StratifiedKFold

import pandas as pd
import matplotlib.pyplot as plt

from foldrm import Classifier
from utils import split_data # Or your stratified version if you prefer
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [16]:
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU found")

GPU available: False
No GPU found


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [18]:
random.seed(42)

# HUMAN THREATS Included

In [19]:
model, data = final_extinctionrisk_dataframe()

categorical_data = data.drop(model.numeric, axis=1)
categorical_data_without_y = categorical_data.drop(model.label, axis=1)
categorical_data_without_y_dummies = pd.get_dummies(categorical_data_without_y, dtype=int)
one_hot_dataset = pd.concat([data[model.numeric], categorical_data_without_y_dummies],axis=1)

X = one_hot_dataset

X = X.to_numpy()
mapping = {'Lower_risk':0, 'Higher_risk':1}
y = data[model.label]
y = y.map(mapping)
y = y.to_numpy()

X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

X.to(device)
y.to(device)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [20]:
def model_train(model, X_train, y_train, X_val, y_val):
    # loss function and optimizer
    loss_fn = nn.BCELoss()  # binary cross entropy
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
 
    n_epochs = 1000   # number of epochs to run
    batch_size = 8  # size of each batch
    batch_start = torch.arange(0, len(X_train), batch_size)
 
    # Hold the best model
    best_acc = - np.inf   # init to negative infinity
    best_rec = - np.inf   # init to negative infinity
    best_pre = - np.inf   # init to negative infinity
    best_f1 = - np.inf   # init to negative infinity
    best_weights = None
 
    for epoch in range(n_epochs):
        model.train()
        with tqdm.tqdm(batch_start, unit="batch", mininterval=0, disable=True) as bar:
            bar.set_description(f"Epoch {epoch}")
            for start in bar:
                # take a batch
                X_batch = X_train[start:start+batch_size]
                y_batch = y_train[start:start+batch_size].unsqueeze(dim=1)
                # forward pass
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
                # backward pass
                optimizer.zero_grad()
                loss.backward()
                # update weights
                optimizer.step()
                # print progress
                acc = (y_pred.round() == y_batch).float().mean()
                bar.set_postfix(
                    loss=float(loss),
                    acc=float(acc)
                )
        # evaluate accuracy at end of each epoch
        model.eval()
        y_pred = model(X_val)
        preds_cpu = y_pred.cpu()
        labels_cpu = y_val.cpu()
        preds_numpy = np.round(preds_cpu.detach().numpy().flatten())
        labels_numpy = labels_cpu.detach().numpy()
        f1 = sklearn_f1_score(labels_numpy, preds_numpy, average=None)
        acc = accuracy_score(labels_numpy, preds_numpy)
        precision = precision_score(labels_numpy, preds_numpy, average=None)
        recall = recall_score(labels_numpy, preds_numpy, average=None)
        
        if acc > best_acc:
            best_acc = acc
            best_weights = copy.deepcopy(model.state_dict())
            best_rec = recall
            best_pre = precision
            best_f1 = f1
            stop_state = epoch
    # restore model and return best accuracy
    model.load_state_dict(best_weights)
    return best_acc, best_rec, best_pre, best_f1, stop_state

In [ ]:
class SLP(torch.nn.Module):
    def __init__(self, input_size):
        super(SLP, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.linear1(x))
        return x
    
# create model, train, and get accuracy
model = SLP(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("Single Layer Perceptron, Including Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

In [ ]:
class TwoLayerMLPwithRELU(torch.nn.Module):
    def __init__(self, input_size):
        super(TwoLayerMLPwithRELU, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 32)
        self.activation1 = torch.nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.linear2 = torch.nn.Linear(32, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.activation1(self.linear1(x))
        x = self.dropout1(x)
        x = self.sigmoid(self.linear2(x))
        return x

# create model, train, and get accuracy
model = TwoLayerMLPwithRELU(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("2 Layer MLP, with dropout Including Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

2 Layer MLP, with dropout Including Human Threats
Accuracy:0.883489784649365
Precision:[0.88654681 0.85858586]
Recall[0.98079561 0.4815864 ]
F1:[0.93129274 0.61705989]
epoch:492


In [ ]:
class FiveLayerMLPwithRELU(torch.nn.Module):
    def __init__(self, input_size):
        super(FiveLayerMLPwithRELU, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 32)
        self.activation1 = torch.nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.linear2 = torch.nn.Linear(32, 32)
        self.activation2 = torch.nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.linear3 = torch.nn.Linear(32, 32)
        self.activation3 = torch.nn.ReLU()
        self.dropout3 = nn.Dropout(0.2)
        self.linear4 = torch.nn.Linear(32, 32)
        self.activation4 = torch.nn.ReLU()
        self.dropout4 = nn.Dropout(0.2)
        self.linear5 = torch.nn.Linear(32, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.activation1(self.linear1(x))
        x = self.dropout1(x)
        x = self.activation2(self.linear2(x))
        x = self.dropout2(x)
        x = self.activation3(self.linear3(x))
        x = self.dropout3(x)
        x = self.activation4(self.linear4(x))
        x = self.dropout4(x)
        x = self.sigmoid(self.linear5(x))
        return x

# create model, train, and get accuracy
model = FiveLayerMLPwithRELU(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("5 Layer MLP,  with Dropout and Including Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


5 Layer MLP,  with Dropout and Including Human Threats
Accuracy:0.8879072335726118
Precision:[0.89292423 0.85046729]
Recall[0.97805213 0.51558074]
F1:[0.93355155 0.64197531]
epoch:497


In [ ]:
const_bnn_prior_parameters = {
        "prior_mu": 0.0,
        "prior_sigma": 1.0,
        "posterior_mu_init": 0.0,
        "posterior_rho_init": -3.0,
        "type": "Reparameterization",  # Flipout or Reparameterization
        "moped_enable": True,  # True to initialize mu/sigma from the pretrained dnn weights
        "moped_delta": 0.5,
}

model = TwoLayerMLPwithRELU(np.shape(X)[1])

dnn_to_bnn(model, const_bnn_prior_parameters)
model.to(device)

model = model
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Accuracy:0.8558807288790723
Precision:[0.86740331 0.75274725]
Recall[0.9691358  0.38810198]
F1:[0.9154519  0.51214953]
epoch:477


# HUMAN THREATS not included

In [ ]:
model, data = final_extinctionrisk_noth_dataframe()

categorical_data = data.drop(model.numeric, axis=1)
categorical_data_without_y = categorical_data.drop(model.label, axis=1)
categorical_data_without_y_dummies = pd.get_dummies(categorical_data_without_y, dtype=int)
one_hot_dataset = pd.concat([data[model.numeric], categorical_data_without_y_dummies],axis=1)

X = one_hot_dataset

X = X.to_numpy()
mapping = {'Lower_risk':0, 'Higher_risk':1}
y = data[model.label]
y = y.map(mapping)
y = y.to_numpy()

X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

X.to(device)
y.to(device)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
# create model, train, and get accuracy
model = SLP(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("Single Layer Perceptron, Without Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

Single Layer Perceptron, Without Human Threats
Accuracy:0.864715626725566
Precision:[0.87600744 0.77272727]
Recall[0.9691358  0.43342776]
F1:[0.92022143 0.5553539 ]
epoch:279


In [ ]:
# create model, train, and get accuracy
model = TwoLayerMLPwithRELU(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("2 Layer MLP, Without Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

2 Layer MLP, Without Human Threats
Accuracy:0.8569850911098841
Precision:[0.86049308 0.81756757]
Recall[0.98148148 0.3427762 ]
F1:[0.91701378 0.48303393]
epoch:363


In [ ]:
# create model, train, and get accuracy
model = FiveLayerMLPwithRELU(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("5 Layer MLP, Without Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

KeyboardInterrupt: 